# JOINTS 2026 External Data Notebook

Objective: Build the external datasets used by the pipeline notebooks and save them to the external/ folder, using only sources made public on or before 30 September 2025, as required by the competition rules.

# Import modules and libraries

In [ ]:
import os, re, csv, sys, calendar, threading
from datetime import datetime
from pathlib import Path
import pandas as pd
try:
    from playwright.sync_api import sync_playwright   # only needed to re-scrape Cinepoint
except ImportError:
    sync_playwright = None


# Output folder
Every dataset is saved as a CSV in the external/ folder at the repo root, which the pipeline notebooks read from.

In [ ]:
EXTERNAL_DIR = "../external"
os.makedirs(EXTERNAL_DIR, exist_ok=True)
print("EXTERNAL_DIR =", os.path.abspath(EXTERNAL_DIR))

# Calendar data

### Cuti bersama
Collective leave days set by the government on top of national holidays, which holidays.csv does not include. One row per date:
- date, name: the leave day and the holiday it belongs to.
- source, published: the decree and its signing date, both on or before 30 September 2025.

In [ ]:
SKB_2025 = "SKB 3 Menteri No. 1017, 2, 2 Tahun 2024"
SKB_2025_AUG = "SKB 3 Menteri No. 933, 1, 3 Tahun 2025"
SKB_2026 = "SKB 3 Menteri No. 1497, 2, 5 Tahun 2025"

cuti_bersama = pd.DataFrame([
    ("2025-01-28", "Tahun Baru Imlek 2576", SKB_2025, "2024-10-14"),
    ("2025-03-28", "Nyepi 1947", SKB_2025, "2024-10-14"),
    ("2025-04-02", "Idulfitri 1446 H", SKB_2025, "2024-10-14"),
    ("2025-04-03", "Idulfitri 1446 H", SKB_2025, "2024-10-14"),
    ("2025-04-04", "Idulfitri 1446 H", SKB_2025, "2024-10-14"),
    ("2025-04-07", "Idulfitri 1446 H", SKB_2025, "2024-10-14"),
    ("2025-05-13", "Waisak 2569 BE", SKB_2025, "2024-10-14"),
    ("2025-05-30", "Kenaikan Yesus Kristus", SKB_2025, "2024-10-14"),
    ("2025-06-09", "Iduladha 1446 H", SKB_2025, "2024-10-14"),
    ("2025-08-18", "Proklamasi Kemerdekaan", SKB_2025_AUG, "2025-08-07"),
    ("2025-12-26", "Kelahiran Yesus Kristus", SKB_2025, "2024-10-14"),
    ("2026-02-16", "Tahun Baru Imlek 2577", SKB_2026, "2025-09-19"),
    ("2026-03-18", "Nyepi 1948", SKB_2026, "2025-09-19"),
    ("2026-03-20", "Idulfitri 1447 H", SKB_2026, "2025-09-19"),
    ("2026-03-23", "Idulfitri 1447 H", SKB_2026, "2025-09-19"),
    ("2026-03-24", "Idulfitri 1447 H", SKB_2026, "2025-09-19"),
    ("2026-05-15", "Kenaikan Yesus Kristus", SKB_2026, "2025-09-19"),
    ("2026-05-28", "Iduladha 1447 H", SKB_2026, "2025-09-19"),
    ("2026-12-24", "Kelahiran Yesus Kristus", SKB_2026, "2025-09-19"),
], columns=["date", "name", "source", "published"])
cuti_bersama["date"] = pd.to_datetime(cuti_bersama["date"])
cuti_bersama

### National holidays before the data
holidays.csv starts on 1 April 2025, so it misses the national holidays just before it that belong to the Lebaran 2025 days-off block (28 March to 7 April 2025). Dates from SKB 3 Menteri No. 1017, 2, 2 Tahun 2024 (signed 14 October 2024):
- date, name: the national holiday.

In [ ]:
extra_national_holidays = pd.DataFrame([
    ("2025-03-29", "Nyepi 1947", SKB_2025, "2024-10-14"),
    ("2025-03-31", "Idulfitri 1446 H", SKB_2025, "2024-10-14"),
], columns=["date", "name", "source", "published"])
extra_national_holidays["date"] = pd.to_datetime(extra_national_holidays["date"])
extra_national_holidays

### Ramadan
The fasting month, when cinema attendance changes a lot, is not marked in holidays.csv. Dates follow Muhammadiyah's Kalender Hijriah Global Tunggal (KHGT), launched publicly on 25 June 2025, which puts 1 Ramadan 1447 H on 18 February 2026 and Idulfitri on 20 March 2026. The later Maklumat PP Muhammadiyah No. 2/MLM/I.0/E/2025 was signed 22 September 2025 but only reported in October 2025, so it is not used as the source. The government date (19 February 2026) was only set by the isbat session in February 2026, after the cutoff. Ramadan 1446 H (March 2025) ends before the train data starts, so it is not included. One row per date:
- date, ramadan_day: the date and the day number of the fasting month.

In [ ]:
RAMADAN_1447_START, IDULFITRI_1447 = pd.Timestamp("2026-02-18"), pd.Timestamp("2026-03-20")

ramadan = pd.DataFrame({"date": pd.date_range(RAMADAN_1447_START, IDULFITRI_1447 - pd.Timedelta(days=1))})
ramadan["ramadan_day"] = range(1, len(ramadan) + 1)
ramadan["source"] = "Kalender Hijriah Global Tunggal (KHGT) Muhammadiyah"
ramadan["published"] = "2025-06-25"
print(f"ramadan days: {len(ramadan)} | {ramadan['date'].min().date()} to {ramadan['date'].max().date()}")
ramadan.head()

### School holidays
School breaks from the DKI Jakarta education calendars (Kalender Pendidikan 2024/2025 and 2025/2026, Keputusan Kepala Dinas Pendidikan Jakarta No. 89 Tahun 2025). Other provinces have different dates and are not added until each document is checked. One row per date:
- date, region, period: the break day, the province, and which break it belongs to.

In [ ]:
SCHOOL_BREAKS = [
    ("2025-03-25", "2025-04-07", "Idulfitri 1446 H", "Kalender Pendidikan DKI 2024/2025", "2024"),
    ("2025-06-28", "2025-07-13", "Kenaikan kelas", "Kalender Pendidikan DKI 2024/2025 dan 2025/2026", "2025-07-11"),
    ("2025-12-22", "2026-01-04", "Semester ganjil", "Kepdis Pendidikan Jakarta No. 89 Tahun 2025", "2025-07-11"),
    ("2026-02-16", "2026-02-20", "Awal Ramadan", "Kepdis Pendidikan Jakarta No. 89 Tahun 2025", "2025-07-11"),
    ("2026-03-16", "2026-03-27", "Idulfitri 1447 H", "Kepdis Pendidikan Jakarta No. 89 Tahun 2025", "2025-07-11"),
]
school_holidays = pd.concat([
    pd.DataFrame({"date": pd.date_range(start, end), "region": "DKI Jakarta", "period": period,
                  "source": source, "published": published})
    for start, end, period, source, published in SCHOOL_BREAKS
], ignore_index=True)
school_holidays.groupby("period", sort=False)["date"].agg(["min", "max", "count"])

# Box office data

### Lebaran admissions (news reports)
Daily and cumulative admissions of the big Lebaran releases of 2024 and 2025, collected from news articles published in April 2024 and April 2025 (data from Cinepoint and the filmmakers). Train starts on Idulfitri 2025, so these numbers are the only record of how Lebaran-week sales move. One row per reported number:
- film, release_date, eid_date: the film, its release date and the Idulfitri date of that year.
- date, day: the date the number refers to and the day since release (release day = 1).
- metric: daily (tickets on that day) or cumulative (total since release).
- admissions, approx: the number, and whether it is a rounded milestone such as "2 juta".
- source, published: the article URL and its publication date.

In [ ]:
KOMPAS_0408 = ("https://www.kompas.com/hype/read/2025/04/08/091334966/perolehan-sementara-penonton-film-lebaran-2025-pabrik-gula-memimpin-jumbo", "2025-04-08")
KOMPAS_0415 = ("https://www.kompas.com/hype/read/2025/04/15/073428966/perolehan-sementara-film-film-lebaran-2025-jumbo-melesat-ke-3-juta-penonton", "2025-04-15")
KOMPAS_0406 = ("https://www.kompas.com/hype/read/2025/04/06/204415166/6-hari-tayang-pabrik-gula-catatkan-2-juta-penonton", "2025-04-06")
IDN_0407 = ("https://www.idntimes.com/hype/entertainment/jumlah-penonton-minggu-pertama-film-indonesia-lebaran-2025-00-19lfc-f2rdkk", "2025-04-07")
CNN_0410 = ("https://www.cnnindonesia.com/hiburan/20250410153809-220-1217492/box-office-10-hari-film-lebaran-2025-pabrik-gula-pertama-jumbo-kedua", "2025-04-10")
LIPUTAN_0412 = ("https://www.liputan6.com/showbiz/read/5571848/badarawuhi-di-desa-penari-raih-344507-penonton-di-hari-pertama-siksa-kubur-meneror-250-ribuan", "2024-04-12")
DISWAY_0418 = ("https://harian.disway.id/read/777493/adu-laris-film-siksa-kubur-dan-badarawuhi-di-desa-penari-tembus-2-juta-penonton-dalam-sepekan", "2024-04-18")

R2025, EID2025 = "2025-03-31", "2025-03-31"
R2024, EID2024 = "2024-04-11", "2024-04-10"

LEBARAN_ROWS = [
    ("PABRIK GULA", R2025, EID2025, "2025-03-31", "daily", 203799, False, IDN_0407),
    ("PABRIK GULA", R2025, EID2025, "2025-04-03", "cumulative", 1000000, True, IDN_0407),
    ("PABRIK GULA", R2025, EID2025, "2025-04-05", "cumulative", 2000000, True, KOMPAS_0406),
    ("PABRIK GULA", R2025, EID2025, "2025-04-06", "cumulative", 2281222, False, IDN_0407),
    ("PABRIK GULA", R2025, EID2025, "2025-04-07", "daily", 311192, False, KOMPAS_0408),
    ("PABRIK GULA", R2025, EID2025, "2025-04-07", "cumulative", 2597861, False, KOMPAS_0408),
    ("PABRIK GULA", R2025, EID2025, "2025-04-09", "cumulative", 3181792, False, CNN_0410),
    ("PABRIK GULA", R2025, EID2025, "2025-04-14", "cumulative", 4028427, False, KOMPAS_0415),
    ("JUMBO", R2025, EID2025, "2025-03-31", "daily", 60458, False, IDN_0407),
    ("JUMBO", R2025, EID2025, "2025-04-06", "cumulative", 1027423, False, IDN_0407),
    ("JUMBO", R2025, EID2025, "2025-04-07", "daily", 326257, False, KOMPAS_0408),
    ("JUMBO", R2025, EID2025, "2025-04-07", "cumulative", 1331509, False, KOMPAS_0408),
    ("JUMBO", R2025, EID2025, "2025-04-10", "cumulative", 1858149, False, CNN_0410),
    ("JUMBO", R2025, EID2025, "2025-04-14", "cumulative", 3528792, False, KOMPAS_0415),
    ("QODRAT 2", R2025, EID2025, "2025-03-31", "daily", 72377, False, IDN_0407),
    ("QODRAT 2", R2025, EID2025, "2025-04-06", "cumulative", 1090299, False, IDN_0407),
    ("QODRAT 2", R2025, EID2025, "2025-04-07", "daily", 207797, False, KOMPAS_0408),
    ("QODRAT 2", R2025, EID2025, "2025-04-07", "cumulative", 1301462, False, KOMPAS_0408),
    ("QODRAT 2", R2025, EID2025, "2025-04-09", "cumulative", 1415568, False, CNN_0410),
    ("QODRAT 2", R2025, EID2025, "2025-04-14", "cumulative", 1928633, False, KOMPAS_0415),
    ("KOMANG", R2025, EID2025, "2025-03-31", "daily", 60218, False, IDN_0407),
    ("KOMANG", R2025, EID2025, "2025-04-06", "cumulative", 801311, False, IDN_0407),
    ("KOMANG", R2025, EID2025, "2025-04-07", "daily", 168960, False, KOMPAS_0408),
    ("KOMANG", R2025, EID2025, "2025-04-07", "cumulative", 926277, False, KOMPAS_0408),
    ("KOMANG", R2025, EID2025, "2025-04-10", "cumulative", 1288952, False, CNN_0410),
    ("KOMANG", R2025, EID2025, "2025-04-14", "cumulative", 2045167, False, KOMPAS_0415),
    ("NORMA: ANTARA MERTUA DAN MENANTU", R2025, EID2025, "2025-03-31", "daily", 29775, False, IDN_0407),
    ("NORMA: ANTARA MERTUA DAN MENANTU", R2025, EID2025, "2025-04-06", "cumulative", 378886, False, IDN_0407),
    ("NORMA: ANTARA MERTUA DAN MENANTU", R2025, EID2025, "2025-04-07", "cumulative", 455825, False, KOMPAS_0408),
    ("NORMA: ANTARA MERTUA DAN MENANTU", R2025, EID2025, "2025-04-10", "cumulative", 561000, True, CNN_0410),
    ("NORMA: ANTARA MERTUA DAN MENANTU", R2025, EID2025, "2025-04-14", "cumulative", 662613, False, KOMPAS_0415),
    ("BADARAWUHI DI DESA PENARI", R2024, EID2024, "2024-04-11", "daily", 344507, False, LIPUTAN_0412),
    ("BADARAWUHI DI DESA PENARI", R2024, EID2024, "2024-04-13", "cumulative", 1031707, False, DISWAY_0418),
    ("BADARAWUHI DI DESA PENARI", R2024, EID2024, "2024-04-16", "cumulative", 2000000, True, DISWAY_0418),
    ("SIKSA KUBUR", R2024, EID2024, "2024-04-11", "daily", 257871, False, DISWAY_0418),
    ("SIKSA KUBUR", R2024, EID2024, "2024-04-14", "cumulative", 1020981, False, DISWAY_0418),
    ("SIKSA KUBUR", R2024, EID2024, "2024-04-17", "cumulative", 2003981, False, DISWAY_0418),
]
lebaran_admissions = pd.DataFrame(
    [(f, r, e, d, m, a, ap, s[0], s[1]) for f, r, e, d, m, a, ap, s in LEBARAN_ROWS],
    columns=["film", "release_date", "eid_date", "date", "metric", "admissions", "approx", "source", "published"],
)
for col in ["release_date", "eid_date", "date"]:
    lebaran_admissions[col] = pd.to_datetime(lebaran_admissions[col])
lebaran_admissions.insert(4, "day", (lebaran_admissions["date"] - lebaran_admissions["release_date"]).dt.days + 1)
assert (pd.to_datetime(lebaran_admissions["published"]) <= "2025-09-30").all()
lebaran_admissions.groupby("film")[["day"]].agg(["min", "max", "count"])

### Lebaran week check
Average daily admissions in each reported window divided by the release-day admissions, to see how strongly sales grow during the Lebaran holiday instead of decaying.

In [ ]:
cum = lebaran_admissions[lebaran_admissions["metric"] == "cumulative"].sort_values(["film", "day"])
first_day = lebaran_admissions[(lebaran_admissions["metric"] == "daily") & (lebaran_admissions["day"] == 1)].set_index("film")["admissions"]
prev = cum.groupby("film")[["day", "admissions"]].shift()
windows = cum.assign(
    from_day=prev["day"].fillna(1).astype(int) + 1,
    window_daily_avg=(cum["admissions"] - prev["admissions"].fillna(cum["film"].map(first_day))) / (cum["day"] - prev["day"].fillna(1)),
)
windows["vs_release_day"] = (windows["window_daily_avg"] / windows["film"].map(first_day)).round(2)
windows[["film", "from_day", "day", "window_daily_avg", "vs_release_day", "approx"]].round({"window_daily_avg": 0})

### Audience statistics
Statements and numbers from the cinema industry about how seasons change attendance, kept only when the page was opened and its publication date read, and the date is on or before 30 September 2025. They are priors for the seasonal postprocessing factors, not model features. One row per statement:
- topic, statement: what is described and the content in plain words (paraphrased, not verbatim).
- value, unit, period_referred: the number if there is one, its unit and the period it refers to.
- speaker_or_source, url, published: who said it, the page, and its publication date.

Leads that were found but are left out because the date or content could not be checked: a survey result of 17.25% of Indonesian-film admissions falling within 14 days around Eid (2007-2024; the only pages found were dated March 2026), SMRC 2019 and Populix 2022 youth surveys (publication date not read), and every article from 2026.

In [ ]:
audience_statistics = pd.DataFrame([
    ("ramadan_effect", "Cineplex 21 said attendance drops 30 to 40 percent during Ramadan, also on weekends, because people go home early to break the fast and rest before dawn.",
     "30-40", "percent", "Ramadan 2013", "Chaterine Keng, corporate secretary Cineplex 21",
     "https://www.kapanlagi.com/showbiz/film/indonesia/ramadan-jumlah-penonton-bioskop-turun-puluhan-persen-dd00be.html", "2013-07-12"),
    ("school_holiday_effect", "GPBSI chair said the industry relies on Eid and year-end holidays, and school breaks do not affect occupancy much because families spend on uniforms and supplies for the new school year.",
     "", "", "school breaks 2023", "Djonny Syafruddin, chair GPBSI",
     "https://industri.kontan.co.id/news/pasca-pandemi-covid-19-industri-bioskop-di-tanah-air-mulai-bangkit", "2023-07-04"),
    ("monthly_admissions", "Cinema XXI had more than 14 million admissions in April 2025 (Lebaran), a monthly record; the previous record was 11.7 million in June 2019.",
     "14000000", "admissions", "April 2025", "Cinema XXI (CNMA)",
     "https://www.bareksa.com/berita/saham/2025-05-02/cinema-xxi-cnma-catat-rekor-jumlah-penonton-capai-14-juta-di-april-2025", "2025-05-02"),
    ("monthly_admissions", "Cinema XXI had about 8 million admissions in May 2025, about 22 million over April and May together.",
     "8000000", "admissions", "May 2025", "Cinema XXI (CNMA)",
     "https://insight.kontan.co.id/news/jumlah-penonton-bioskop-cinema-xxi-membludak-kinerja-cnma-diproyeksi-bakal-meningkat", "2025-06-19"),
    ("quarterly_admissions", "Cinema XXI had 25 million admissions in the second quarter of 2024.",
     "25000000", "admissions", "Q2 2024", "Cinema XXI (CNMA)",
     "https://insight.kontan.co.id/news/jumlah-penonton-bioskop-cinema-xxi-membludak-kinerja-cnma-diproyeksi-bakal-meningkat", "2025-06-19"),
    ("annual_admissions", "Cinema XXI had 84 million admissions in 2024 and targeted 95 million for 2025.",
     "84000000", "admissions", "2024", "Cinema XXI (CNMA)",
     "https://insight.kontan.co.id/news/jumlah-penonton-bioskop-cinema-xxi-membludak-kinerja-cnma-diproyeksi-bakal-meningkat", "2025-06-19"),
    ("ramadan_effect", "Cinema XXI ticket revenue fell 30 percent year on year to Rp582 billion in Q1 2025; moviegoing dropped in March 2025 (Ramadan) compared with usual days, and April 2025 admissions were already close to the whole Q1 total.",
     "-30", "percent (ticket revenue YoY)", "Q1 2025", "Cinema XXI (CNMA), reported by Indo Premier",
     "https://www.indopremier.com/ipotnews/newsDetail.php?jdl=Cinema_XXI_(CNMA)_Rugi_di_Kuartal_I_2025_Imbas_Penonton_Turun_saat_Ramadan&news_id=462862&group_news=RESEARCHNEWS&news_date=&taging_subtype=CNMA&name=&search=y_general&q=Nusantara+Sejahtera+Raya&halaman=1", "2025-04-25"),
    ("film_supply", "Cinema XXI had five films above 1 million admissions in Q1 2025 against seven in Q1 2024.",
     "5", "films above 1 million", "Q1 2025", "Cinema XXI (CNMA)",
     "https://www.bareksa.com/berita/saham/2025-05-02/cinema-xxi-cnma-catat-rekor-jumlah-penonton-capai-14-juta-di-april-2025", "2025-05-02"),
], columns=["topic", "statement", "value", "unit", "period_referred", "speaker_or_source", "url", "published"])
audience_statistics["published"] = pd.to_datetime(audience_statistics["published"])
assert (audience_statistics["published"] <= "2025-09-30").all()
audience_statistics[["topic", "period_referred", "value", "unit", "published"]]

### Cinepoint daily top box office
Real national admissions per film and day from cinepoint.com (public page "Top Box Office", daily view). The page lists a ranking per date (rank, title, daily admissions, cumulative admissions, showtimes); the site API rejects direct calls, so the scraper drives a real Chrome window with Playwright, picks the year, month and 100 rows per page, opens each date and reads the popup. It is resumable: dates already in the CSV are skipped.

- `cinepoint_daily_top_prev.csv`: 2023-09-20 to 2025-03-31 (previous Christmas, Ramadan and Lebaran seasons), before the cutoff; Cinepoint lists only about 5 films per day before 2024-09.
- `cinepoint_daily_top_train.csv`: 2025-04-01 to 2025-09-30, before the competition cutoff, usable as external data.
- `cinepoint_daily_top.csv`: 2025-10-01 to 2026-03-31, after the 2025-09-30 cutoff, so anything built on it is a trial / assumption.
- Scraped on 2026-10-08. Small films drop out of the daily list on later days, which is kept as information (`cutK` in `main.ipynb`).
- 2026-10-08 fix: the day popup shows 10 films per page; the scraper now sets its own "Rows per page" to the largest option and only writes a day when the rows read equal the total the popup reports (the old paging re-read stale pages).

In [ ]:
CINEPOINT_DIR = os.path.join(EXTERNAL_DIR, "cinepoint")
os.makedirs(CINEPOINT_DIR, exist_ok=True)
# the scraper lives in external/cinepoint/cinepoint_scrape.py (also runnable from the command line); imported here so there is one copy
sys.path.insert(0, CINEPOINT_DIR)
try:
    from cinepoint_scrape import scrape_cinepoint
except ImportError:   # playwright not installed: the CSVs below can still be read
    scrape_cinepoint = None

In [ ]:
SCRAPE_CINEPOINT = False   # needs a visible Chrome window and about 35 s per date, so the CSVs below are only rebuilt on request
CINEPOINT_RUNS = [
    ("2023-09-20", "2025-03-31", "cinepoint_daily_top_prev.csv"),   # previous seasons, before the cutoff (timing shape in main_legit)
    ("2025-04-01", "2025-09-30", "cinepoint_daily_top_train.csv"),   # train period, before the cutoff
    ("2025-10-01", "2026-03-31", "cinepoint_daily_top.csv"),        # test period, after the cutoff: trial / assumption only
]
if SCRAPE_CINEPOINT and scrape_cinepoint is not None:
    for start, end, name in CINEPOINT_RUNS:
        # Playwright's sync API cannot run inside the notebook's event loop, so it gets its own thread
        worker = threading.Thread(target=scrape_cinepoint, args=(start, end, os.path.join(CINEPOINT_DIR, name)))
        worker.start()
        worker.join()

for _, _, name in CINEPOINT_RUNS:
    path = os.path.join(CINEPOINT_DIR, name)
    if os.path.exists(path):
        daily_top = pd.read_csv(path, parse_dates=["date"])
        print(f"{name}: {len(daily_top)} rows, {daily_top['date'].nunique()} dates, {daily_top['date'].min().date()} to {daily_top['date'].max().date()}")

### Test films
One row per test film (format variants kept apart) with the figures from `test_history.csv` and `test.csv`, a reference list for checking the Cinepoint match.

In [ ]:
test_hist = pd.read_csv("../data/test_history.csv", parse_dates=["date_show"])
test_rows = pd.read_csv("../data/test.csv", parse_dates=["date_show"])
test_films = (test_hist.groupby("movie_title")
              .agg(d1=("date_show", "min"), d13_tickets=("total_ticket", "sum"), cinemas=("cinema_ids", "nunique"))
              .reset_index())
test_films["d4_date"] = test_films["d1"] + pd.Timedelta(days=3)
test_films["d10_date"] = test_films["d1"] + pd.Timedelta(days=9)
test_films = test_films.merge(test_rows.groupby("movie_title").size().rename("test_rows").reset_index(), on="movie_title", how="left")
os.makedirs(os.path.join(EXTERNAL_DIR, "test_movies"), exist_ok=True)
test_films.sort_values("movie_title").to_csv(os.path.join(EXTERNAL_DIR, "test_movies", "test_films.csv"), index=False)
test_films.head()

# Save datasets

In [ ]:
DATASETS = {"cuti_bersama": cuti_bersama, "extra_national_holidays": extra_national_holidays, "ramadan": ramadan, "school_holidays": school_holidays,
            "lebaran_admissions": lebaran_admissions,
            "audience_statistics": audience_statistics}
for name, df in DATASETS.items():
    df.to_csv(f"{EXTERNAL_DIR}/{name}.csv", index=False)
    print(f"{name}.csv: {df.shape}")